In [1]:
import os

import pandas as pd

In [2]:
data_dir = '/Users/rebekahzhang/data/neural_data/processing_check'

progress_file_name = "RZ_progress.csv"  # downloaded from df pipeline by running progress_checker.py
recording_log_name = "recording_log.csv"  # downloaded from recording log from google drive

### dj progress check

In [15]:
dj_progress = pd.read_csv(os.path.join(data_dir, progress_file_name))

In [16]:
dj_progress

,subject,session_datetime,insertion_number,Event,Synchronization,ProbeInsertion,ClusteringTask,PreProcessing,SIClustering,PostProcessing,SIExport,Clustering,CuratedClustering,ManualCuration,OfficialCuration,ApplyOfficialCuration
0,RZ034,2024-07-11 13:31:24,0,Done,Done,Done,Done,Done,X,X,X,X,X,X,X,X
1,RZ034,2024-07-13 12:58:26,0,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,X,X,X
2,RZ034,2024-07-13 12:58:26,1,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done
3,RZ034,2024-07-14 12:52:46,0,Done,Done,Done,Done,Done,Done,Done,X,Done,Done,X,X,X
4,RZ034,2024-07-14 12:52:46,1,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,RZ070,2025-02-12 14:02:10,1,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done
126,RZ070,2025-02-13 11:40:15,0,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done
127,RZ070,2025-02-13 11:40:15,1,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done
128,RZ070,2025-02-14 11:03:49,0,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done,Done


In [17]:
flow_columns = [
    'Event', 'Synchronization', 'ProbeInsertion', 'ClusteringTask', 
    'PreProcessing', 'SIClustering', 'PostProcessing', 'SIExport', 
    'Clustering', 'CuratedClustering', 'ManualCuration', 'OfficialCuration', 
    'ApplyOfficialCuration'
]
dj_progress['First_X_Column'] = dj_progress[flow_columns].eq('X').idxmax(axis=1)
dj_progress['First_X_Column'] = dj_progress['First_X_Column'].replace('Event', 'Done')

In [18]:
counts = dj_progress['First_X_Column'].value_counts()
print(counts)

First_X_Column
Done              104
SIClustering       11
SIExport           10
ManualCuration      5
Name: count, dtype: int64


adding progress info to recording log

In [19]:
recording_log = pd.read_csv(os.path.join(data_dir, recording_log_name))
recording_log = recording_log.drop(columns=['NIDAQ', 'simultaneous',
       'probe', 'hemisphere', 'depth', 'probe treatment', 'insertion speed',
       'resting time', 'surface', 'extraction speed', 'notes', 'rewards',
       'num trials', 'tw', 'current status', 'final data out'])

In [20]:
recording_log

,date,mouse,insertion_number,region,potential problems,sorting notes
0,2024-05-10,RZ043,0,v1,NaN,exp3
1,2024-05-10,RZ043,1,str,NaN,exp3
2,2024-05-25,RZ044,0,v1,NaN,exp3
3,2024-05-25,RZ044,1,str,NaN,exp3
4,2024-05-26,RZ044,0,v1,NaN,exp3
...,...,...,...,...,...,...
167,2025-03-19,RZ059,1,str,NaN,NaN
168,2025-03-20,RZ059,0,v1,NaN,"regular sorting, 79 good units; attribute error."
169,2025-03-20,RZ059,1,str,NaN,NaN
170,2025-03-21,RZ059,0,v1,NaN,missing cluster_si_unit_id.tsv. curation delet...


In [21]:
dj_progress['date'] = pd.to_datetime(dj_progress['session_datetime']).dt.date
dj_progress['mouse'] = dj_progress['subject']

In [22]:
# Convert 'date' to datetime (handles strings like '2023-10-01' or mixed formats)
recording_log['date'] = pd.to_datetime(recording_log['date'], errors='coerce')
dj_progress['date'] = pd.to_datetime(dj_progress['date'], errors='coerce')

In [23]:
# Convert to integer (if no decimals)
recording_log['insertion_number'] = recording_log['insertion_number'].astype(int)
dj_progress['insertion_number'] = dj_progress['insertion_number'].astype(int)

In [24]:
# Merge with left join on 3 key columns
merged_df = recording_log.merge(
    dj_progress[['mouse', 'date', 'insertion_number', 'First_X_Column']],
    on=['mouse', 'date', 'insertion_number'],
    how='left'
)

# Label unmatched rows
merged_df['First_X_Column'] = merged_df['First_X_Column'].fillna('not_uploaded')

In [25]:
merged_df

,date,mouse,insertion_number,region,potential problems,sorting notes,First_X_Column
0,2024-05-10,RZ043,0,v1,NaN,exp3,not_uploaded
1,2024-05-10,RZ043,1,str,NaN,exp3,not_uploaded
2,2024-05-25,RZ044,0,v1,NaN,exp3,not_uploaded
3,2024-05-25,RZ044,1,str,NaN,exp3,not_uploaded
4,2024-05-26,RZ044,0,v1,NaN,exp3,not_uploaded
...,...,...,...,...,...,...,...
167,2025-03-19,RZ059,1,str,NaN,NaN,Done
168,2025-03-20,RZ059,0,v1,NaN,"regular sorting, 79 good units; attribute error.",Done
169,2025-03-20,RZ059,1,str,NaN,NaN,Done
170,2025-03-21,RZ059,0,v1,NaN,missing cluster_si_unit_id.tsv. curation delet...,Done


In [39]:
merged_df.to_csv(os.path.join(data_dir, 'sessions_cross_checked.csv'))

### loading cross check results

In [48]:
sessions_cross_checked = pd.read_csv(os.path.join(data_dir, 'sessions_cross_checked.csv'), index_col=0)

In [49]:
progress_overview = sessions_cross_checked['First_X_Column'].value_counts()
print(progress_overview)

First_X_Column
Done              104
not_uploaded       42
SIClustering       11
SIExport           10
ManualCuration      5
Name: count, dtype: int64


In [50]:
sessions_cross_checked

,date,mouse,insertion_number,region,potential problems,sorting notes,First_X_Column
0,2024-05-10,RZ043,0,v1,NaN,exp3,not_uploaded
1,2024-05-10,RZ043,1,str,NaN,exp3,not_uploaded
2,2024-05-25,RZ044,0,v1,NaN,exp3,not_uploaded
3,2024-05-25,RZ044,1,str,NaN,exp3,not_uploaded
4,2024-05-26,RZ044,0,v1,NaN,exp3,not_uploaded
...,...,...,...,...,...,...,...
167,2025-03-19,RZ059,1,str,NaN,NaN,Done
168,2025-03-20,RZ059,0,v1,NaN,"regular sorting, 79 good units; attribute error.",Done
169,2025-03-20,RZ059,1,str,NaN,NaN,Done
170,2025-03-21,RZ059,0,v1,NaN,missing cluster_si_unit_id.tsv. curation delet...,Done


In [54]:
# For each row in the previous cell's DataFrame (sessions_cross_checked[sessions_cross_checked['First_X_Column'] == 'SIClustering']),
# get the corresponding info from dj_progress as a dict

rows = sessions_cross_checked[sessions_cross_checked['First_X_Column'] == 'SIClustering']
result = []
for _, row in rows.iterrows():
    match = dj_progress[
        (dj_progress['mouse'] == row['mouse']) &
        (dj_progress['date'] == pd.to_datetime(row['date'])) &
        (dj_progress['insertion_number'] == row['insertion_number'])
    ]
    if not match.empty:
        info = match.iloc[0][['subject', 'session_datetime', 'insertion_number']].to_dict()
        result.append(info)

result

[{'subject': 'RZ034',
  'session_datetime': '2024-07-11 13:31:24',
  'insertion_number': 0},
 {'subject': 'RZ036',
  'session_datetime': '2024-07-14 14:34:59',
  'insertion_number': 0},
 {'subject': 'RZ038',
  'session_datetime': '2024-07-18 15:07:36',
  'insertion_number': 0},
 {'subject': 'RZ053',
  'session_datetime': '2024-10-24 14:12:22',
  'insertion_number': 1},
 {'subject': 'RZ052',
  'session_datetime': '2024-10-25 11:22:08',
  'insertion_number': 1},
 {'subject': 'RZ053',
  'session_datetime': '2024-10-25 13:19:39',
  'insertion_number': 1},
 {'subject': 'RZ055',
  'session_datetime': '2024-10-29 13:48:08',
  'insertion_number': 0},
 {'subject': 'RZ055',
  'session_datetime': '2024-10-29 13:48:08',
  'insertion_number': 1},
 {'subject': 'RZ055',
  'session_datetime': '2024-11-01 11:04:34',
  'insertion_number': 1},
 {'subject': 'RZ065',
  'session_datetime': '2025-02-22 13:03:06',
  'insertion_number': 0},
 {'subject': 'RZ062',
  'session_datetime': '2025-03-07 11:11:35',
  '

In [55]:
# For each row in the previous cell's DataFrame (sessions_cross_checked[sessions_cross_checked['First_X_Column'] == 'SIClustering']),
# get the corresponding info from dj_progress as a dict

rows = sessions_cross_checked[sessions_cross_checked['First_X_Column'] == 'SIExport']

result = []
for _, row in rows.iterrows():
    match = dj_progress[
        (dj_progress['mouse'] == row['mouse']) &
        (dj_progress['date'] == pd.to_datetime(row['date'])) &
        (dj_progress['insertion_number'] == row['insertion_number'])
    ]
    if not match.empty:
        info = match.iloc[0][['subject', 'session_datetime', 'insertion_number']].to_dict()
        result.append(info)

result

[{'subject': 'RZ034',
  'session_datetime': '2024-07-14 12:52:46',
  'insertion_number': 0},
 {'subject': 'RZ038',
  'session_datetime': '2024-07-16 14:36:56',
  'insertion_number': 0},
 {'subject': 'RZ038',
  'session_datetime': '2024-07-16 14:36:56',
  'insertion_number': 1},
 {'subject': 'RZ037',
  'session_datetime': '2024-07-17 17:09:40',
  'insertion_number': 0},
 {'subject': 'RZ049',
  'session_datetime': '2024-10-31 14:50:15',
  'insertion_number': 0},
 {'subject': 'RZ055',
  'session_datetime': '2024-10-31 10:47:38',
  'insertion_number': 0},
 {'subject': 'RZ049',
  'session_datetime': '2024-11-01 13:55:29',
  'insertion_number': 0},
 {'subject': 'RZ047',
  'session_datetime': '2024-11-19 09:51:41',
  'insertion_number': 0},
 {'subject': 'RZ050',
  'session_datetime': '2024-11-19 12:01:27',
  'insertion_number': 1},
 {'subject': 'RZ047',
  'session_datetime': '2024-11-20 11:55:01',
  'insertion_number': 1}]